# ETF Data — Exploratory Data Analysis

This notebook performs EDA on two Snowflake tables:
- **`ETF_DB.LOCAL_COPY.CONSTITUENTS`** — individual holdings per ETF per date (1.17M rows, 4 snapshots: Feb–Apr 2025)
- **`ETF_DB.LOCAL_COPY.INDUSTRY`** — fund-level metadata per ETF (4,152 ETFs, snapshot 2025-04-01)

Run with:
```
uv run --with 'jupyter,pandas,matplotlib,seaborn,plotly,snowflake-connector-python' jupyter notebook ETF_DB/etf_eda.ipynb
```

## 0. Setup & Data Loading

In [ ]:
import pathlib
import tomllib
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import snowflake.connector

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

# ── Snowflake connection ────────────────────────────────────────────────────
config_path = pathlib.Path.home() / '.snowflake' / 'config.toml'
with open(config_path, 'rb') as f:
    cfg = tomllib.load(f)['connections']['snowconn']

conn = snowflake.connector.connect(
    account=cfg['account'],
    user=cfg['user'],
    password=cfg['password'],
    role=cfg.get('role', 'ACCOUNTADMIN'),
    warehouse=cfg.get('warehouse', 'cortex_analyst_wh'),
)

def query(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

print(f"Connected: {cfg['account']} | {cfg['user']}")

In [ ]:
# Load INDUSTRY table (4,152 rows — fits comfortably in memory)
ind = query("SELECT * FROM ETF_DB.LOCAL_COPY.INDUSTRY")
ind.columns = ind.columns.str.lower()

# Load CONSTITUENTS for the latest snapshot date only
con = query("""
    SELECT *
    FROM ETF_DB.LOCAL_COPY.CONSTITUENTS
    WHERE AS_OF_DATE = (SELECT MAX(AS_OF_DATE) FROM ETF_DB.LOCAL_COPY.CONSTITUENTS)
""")
con.columns = con.columns.str.lower()

# Load all 4 snapshots for time-series analysis (lightweight aggregation in SQL)
con_ts = query("""
    SELECT as_of_date,
           composite_ticker,
           COUNT(*)             AS num_holdings,
           SUM(weight)          AS total_weight,
           MAX(weight)          AS max_weight,
           SUM(market_value)    AS total_market_value
    FROM ETF_DB.LOCAL_COPY.CONSTITUENTS
    GROUP BY 1, 2
    ORDER BY 1, 2
""")
con_ts.columns = con_ts.columns.str.lower()

print(f"INDUSTRY : {ind.shape[0]:,} rows × {ind.shape[1]} cols")
print(f"CONSTITUENTS (latest): {con.shape[0]:,} rows × {con.shape[1]} cols")
print(f"CONSTITUENTS (time-series agg): {con_ts.shape[0]:,} rows")

## 1. INDUSTRY Table — Fund Universe Overview

In [ ]:
print("=== INDUSTRY: shape & dtypes ===")
print(ind.dtypes.to_string())
print(f"\nSnapshot date: {ind['as_of_date'].unique()}")

In [ ]:
# Null / missing value heatmap
null_pct = (ind.isnull().mean() * 100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 6))
null_pct[null_pct > 0].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('% Missing')
ax.set_title('INDUSTRY — % Missing Values by Column')
plt.tight_layout()
plt.show()
print(null_pct[null_pct > 0].to_string())

In [ ]:
# ETF count by asset class
ac = ind['asset_class'].fillna('Unknown').value_counts().reset_index()
ac.columns = ['asset_class', 'count']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# bar
axes[0].barh(ac['asset_class'], ac['count'], color=sns.color_palette('tab10', len(ac)))
axes[0].set_xlabel('Number of ETFs')
axes[0].set_title('ETF Count by Asset Class')
axes[0].invert_yaxis()
# pie
axes[1].pie(ac['count'], labels=ac['asset_class'], autopct='%1.1f%%',
            colors=sns.color_palette('tab10', len(ac)))
axes[1].set_title('ETF Universe Composition')
plt.suptitle('Fund Universe by Asset Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.1 AUM Analysis

In [ ]:
aum = ind[ind['aum'].notna() & (ind['aum'] > 0)].copy()
aum['aum_bn'] = aum['aum'] / 1e9

print(f"ETFs with AUM data: {len(aum):,} / {len(ind):,}")
print(f"Total AUM : ${aum['aum_bn'].sum():,.1f}B")
print(f"Median AUM: ${aum['aum_bn'].median():,.3f}B")
print(f"Mean AUM  : ${aum['aum_bn'].mean():,.3f}B")
print(f"Max AUM   : ${aum['aum_bn'].max():,.1f}B  ({aum.loc[aum['aum_bn'].idxmax(), 'composite_ticker']})") 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-scale histogram
axes[0].hist(aum['aum_bn'], bins=60, color='steelblue', edgecolor='white', log=True)
axes[0].set_xlabel('AUM ($B)')
axes[0].set_ylabel('Number of ETFs (log scale)')
axes[0].set_title('AUM Distribution (log y-axis)')

# Log-log for power-law inspection
import numpy as np
axes[1].hist(np.log10(aum['aum_bn'] + 1e-6), bins=60, color='darkorange', edgecolor='white')
axes[1].set_xlabel('log₁₀(AUM $B)')
axes[1].set_title('AUM Distribution (log₁₀ scale)')
ticks = [-3, -2, -1, 0, 1, 2, 3]
axes[1].set_xticks(ticks)
axes[1].set_xticklabels([f'$10^{{{t}}}$B' for t in ticks])

plt.suptitle('ETF AUM Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 ETFs by AUM
top_etfs = aum.nlargest(15, 'aum_bn')[['composite_ticker', 'description', 'issuer', 'asset_class', 'aum_bn']]
top_etfs['description'] = top_etfs['description'].str[:40]
top_etfs['aum_bn'] = top_etfs['aum_bn'].round(1)
print("Top 15 ETFs by AUM ($B)")
print(top_etfs.to_string(index=False))

In [ ]:
# Top 10 issuers — AUM and ETF count
issuer = (aum.groupby('issuer')
          .agg(etf_count=('composite_ticker', 'count'),
               total_aum_bn=('aum_bn', 'sum'))
          .sort_values('total_aum_bn', ascending=False)
          .head(10)
          .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = sns.color_palette('tab10', 10)
bars1 = axes[0].barh(issuer['issuer'][::-1], issuer['total_aum_bn'][::-1], color=colors)
axes[0].set_xlabel('Total AUM ($B)')
axes[0].set_title('Top 10 Issuers by Total AUM')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}B'))

bars2 = axes[1].barh(issuer['issuer'][::-1], issuer['etf_count'][::-1], color=colors)
axes[1].set_xlabel('Number of ETFs')
axes[1].set_title('Top 10 Issuers by ETF Count')

plt.suptitle('ETF Issuer Landscape', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# AUM concentration: what % of total AUM do top N ETFs hold?
sorted_aum = aum.sort_values('aum_bn', ascending=False)['aum_bn']
cum_pct = (sorted_aum.cumsum() / sorted_aum.sum() * 100).values

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(cum_pct)+1), cum_pct, color='steelblue')
for n, label in [(10, '10'), (50, '50'), (100, '100'), (200, '200')]:
    if n <= len(cum_pct):
        ax.axvline(n, color='gray', linestyle='--', linewidth=0.8)
        ax.text(n+5, cum_pct[n-1]-3, f'Top {n}\n{cum_pct[n-1]:.1f}%', fontsize=8)
ax.set_xlabel('Number of ETFs (ranked by AUM)')
ax.set_ylabel('Cumulative % of Total AUM')
ax.set_title('AUM Concentration Curve')
plt.tight_layout()
plt.show()

### 1.2 Fee Analysis

In [ ]:
fees = ind[ind['net_expenses'].notna() & (ind['net_expenses'] > 0)].copy()
fees['net_expenses_bps'] = fees['net_expenses'] * 100  # convert to basis points

print(f"ETFs with fee data : {len(fees):,}")
print(f"Min expense ratio  : {fees['net_expenses_bps'].min():.2f} bps")
print(f"Median expense ratio: {fees['net_expenses_bps'].median():.2f} bps")
print(f"Mean expense ratio  : {fees['net_expenses_bps'].mean():.2f} bps")
print(f"Max expense ratio   : {fees['net_expenses_bps'].max():.2f} bps")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(fees['net_expenses_bps'].clip(upper=200), bins=60,
             color='teal', edgecolor='white')
axes[0].axvline(fees['net_expenses_bps'].median(), color='red',
                linestyle='--', label=f"Median: {fees['net_expenses_bps'].median():.0f} bps")
axes[0].set_xlabel('Net Expense Ratio (bps, clipped at 200)')
axes[0].set_ylabel('Number of ETFs')
axes[0].set_title('Expense Ratio Distribution')
axes[0].legend()

# By asset class
fee_ac = (fees.groupby('asset_class')['net_expenses_bps']
          .median().sort_values().reset_index())
axes[1].barh(fee_ac['asset_class'], fee_ac['net_expenses_bps'],
             color=sns.color_palette('tab10', len(fee_ac)))
axes[1].set_xlabel('Median Net Expense Ratio (bps)')
axes[1].set_title('Median Expense Ratio by Asset Class')

plt.suptitle('ETF Fee Structure', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 categories by median fee
fee_cat = (fees.groupby('category')['net_expenses_bps']
           .agg(['median', 'count'])
           .query('count >= 5')
           .sort_values('median', ascending=False)
           .head(15)
           .reset_index())

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fee_cat['category'][::-1], fee_cat['median'][::-1], color='darkorange')
ax.set_xlabel('Median Net Expense Ratio (bps)')
ax.set_title('Top 15 ETF Categories by Median Expense Ratio\n(min 5 ETFs per category)')
plt.tight_layout()
plt.show()

In [ ]:
# AUM vs expense ratio scatter
fee_aum = fees[fees['aum'].notna() & (fees['aum'] > 0)].copy()
fee_aum['aum_bn'] = fee_aum['aum'] / 1e9

fig = px.scatter(
    fee_aum, x='aum_bn', y='net_expenses_bps',
    color='asset_class', hover_data=['composite_ticker', 'description', 'issuer'],
    log_x=True, opacity=0.5, size_max=8,
    labels={'aum_bn': 'AUM ($B, log)', 'net_expenses_bps': 'Net Expense Ratio (bps)'},
    title='AUM vs Expense Ratio by Asset Class'
)
fig.update_traces(marker_size=5)
fig.show()

### 1.3 Geographic & Sector Exposure

In [ ]:
# Region breakdown
region = ind['region'].fillna('Unknown').value_counts().head(15).reset_index()
region.columns = ['region', 'count']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(region['region'][::-1], region['count'][::-1],
             color=sns.color_palette('tab20', len(region)))
axes[0].set_xlabel('Number of ETFs')
axes[0].set_title('ETF Count by Target Region')

# Development class
dev = ind['development_class'].fillna('Unknown').value_counts().reset_index()
dev.columns = ['class', 'count']
axes[1].pie(dev['count'], labels=dev['class'], autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(dev)))
axes[1].set_title('Market Development Class')

plt.suptitle('Geographic Exposure', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Sector exposure (equity ETFs)
sector = (ind[ind['asset_class'] == 'Equity']['sector_exposure']
          .fillna('Broad / Multi-Sector')
          .value_counts()
          .head(20)
          .reset_index())
sector.columns = ['sector', 'count']

fig = px.bar(sector, x='count', y='sector', orientation='h',
             color='count', color_continuous_scale='Blues',
             title='Top 20 Sector Exposures (Equity ETFs)')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

### 1.4 Fund Characteristics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Leveraged
lev = ind['is_leveraged'].value_counts()
axes[0].pie(lev.values, labels=['Non-Leveraged', 'Leveraged'],
            autopct='%1.1f%%', colors=['steelblue', 'tomato'])
axes[0].set_title('Leveraged ETFs')

# ETN vs ETF
etn = ind['is_etn'].value_counts()
axes[1].pie(etn.values, labels=['ETF', 'ETN'],
            autopct='%1.1f%%', colors=['steelblue', 'darkorange'])
axes[1].set_title('ETF vs ETN')

# Distribution frequency
dist = ind['distribution_frequency'].fillna('Unknown').value_counts().head(6)
axes[2].bar(dist.index, dist.values, color=sns.color_palette('tab10', len(dist)))
axes[2].set_title('Distribution Frequency')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('Fund Characteristics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Leveraged ETFs — leverage amount distribution
lev_df = ind[ind['is_leveraged'] == True].copy()
print(f"Leveraged ETF count : {len(lev_df)}")
print(lev_df['leveraged_amount'].value_counts().head(10))

In [ ]:
# Holdings count distribution from INDUSTRY.num_holdings
nh = ind[ind['num_holdings'].notna() & (ind['num_holdings'] > 0)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(nh['num_holdings'].clip(upper=1000), bins=60, color='mediumseagreen', edgecolor='white')
axes[0].set_xlabel('Number of Holdings (clipped at 1000)')
axes[0].set_ylabel('Number of ETFs')
axes[0].set_title('Holdings Count Distribution (INDUSTRY)')

# By asset class
nh_ac = nh.groupby('asset_class')['num_holdings'].median().sort_values().reset_index()
axes[1].barh(nh_ac['asset_class'], nh_ac['num_holdings'],
             color=sns.color_palette('tab10', len(nh_ac)))
axes[1].set_xlabel('Median Number of Holdings')
axes[1].set_title('Median Holdings by Asset Class')

plt.suptitle('Portfolio Breadth', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. CONSTITUENTS Table — Holdings Analysis

In [ ]:
print("=== CONSTITUENTS (latest snapshot) ===")
print(f"Shape       : {con.shape}")
print(f"Snapshot    : {con['as_of_date'].iloc[0]}")
print(f"ETFs        : {con['composite_ticker'].nunique():,}")
print(f"Securities  : {con['constituent_ticker'].nunique():,}")
print(f"\nWeight stats:")
print(con['weight'].describe())

In [ ]:
# Missing values
null_pct_con = (con.isnull().mean() * 100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
null_pct_con[null_pct_con > 0].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('% Missing')
ax.set_title('CONSTITUENTS — % Missing Values by Column')
plt.tight_layout()
plt.show()

### 2.1 Holdings per ETF

In [ ]:
holdings_per_etf = con.groupby('composite_ticker').size().rename('n_holdings')

print(f"Holdings per ETF:")
print(holdings_per_etf.describe())
print(f"\nETFs with 1 holding  : {(holdings_per_etf == 1).sum()}")
print(f"ETFs with 2-10       : {((holdings_per_etf >= 2) & (holdings_per_etf <= 10)).sum()}")
print(f"ETFs with 10-100     : {((holdings_per_etf > 10) & (holdings_per_etf <= 100)).sum()}")
print(f"ETFs with 100-500    : {((holdings_per_etf > 100) & (holdings_per_etf <= 500)).sum()}")
print(f"ETFs with 500+       : {(holdings_per_etf > 500).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(holdings_per_etf.clip(upper=500), bins=60, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Holdings per ETF (clipped at 500)')
axes[0].set_ylabel('Number of ETFs')
axes[0].set_title('Holdings per ETF Distribution')
axes[0].axvline(holdings_per_etf.median(), color='red', linestyle='--',
                label=f"Median: {holdings_per_etf.median():.0f}")
axes[0].legend()

# log scale
import numpy as np
axes[1].hist(np.log10(holdings_per_etf + 1), bins=60, color='darkorange', edgecolor='white')
axes[1].set_xlabel('log₁₀(Holdings per ETF)')
axes[1].set_title('Holdings per ETF (log₁₀ scale)')

plt.suptitle('Portfolio Breadth — Actual Holdings Count', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top & bottom 10 by actual holdings count
top10 = holdings_per_etf.nlargest(10).reset_index()
top10.columns = ['etf', 'holdings']
print("Top 10 ETFs by number of holdings:")
print(top10.to_string(index=False))

### 2.2 Weight Distribution & Concentration

In [ ]:
# Weight distribution (exclude zeros/nulls)
w = con[con['weight'].notna() & (con['weight'] > 0)]['weight']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(w.clip(upper=5), bins=80, color='mediumseagreen', edgecolor='white')
axes[0].set_xlabel('Weight % (clipped at 5%)')
axes[0].set_title('Constituent Weight Distribution')
axes[0].set_ylabel('Count')

axes[1].hist(np.log10(w + 1e-6), bins=80, color='purple', edgecolor='white')
axes[1].set_xlabel('log₁₀(Weight %)')
axes[1].set_title('Constituent Weight (log₁₀ scale)')

plt.suptitle('Holding Weight Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Concentration: top-10 weight sum per ETF (HHI proxy)
etf_concentration = (
    con.sort_values(['composite_ticker', 'weight'], ascending=[True, False])
    .groupby('composite_ticker')
    .head(10)
    .groupby('composite_ticker')['weight']
    .sum()
    .rename('top10_weight')
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(etf_concentration, bins=60, color='steelblue', edgecolor='white')
ax.axvline(etf_concentration.median(), color='red', linestyle='--',
           label=f"Median: {etf_concentration.median():.1f}%")
ax.set_xlabel('Sum of Top-10 Holdings Weight (%)')
ax.set_ylabel('Number of ETFs')
ax.set_title('Portfolio Concentration — Top-10 Holdings Weight Sum')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Median top-10 concentration : {etf_concentration.median():.1f}%")
print(f"ETFs where top-10 > 50%     : {(etf_concentration > 50).sum():,}")
print(f"ETFs where top-10 > 80%     : {(etf_concentration > 80).sum():,}")

### 2.3 Most Widely Held Securities

In [ ]:
# Stocks appearing in most ETFs
ubiquity = (
    con.groupby(['constituent_ticker', 'constituent_name'])
    .agg(
        etf_count=('composite_ticker', 'nunique'),
        total_weight=('weight', 'sum'),
        avg_weight=('weight', 'mean'),
        total_market_value=('market_value', 'sum'),
    )
    .sort_values('etf_count', ascending=False)
    .head(20)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(ubiquity['constituent_ticker'][::-1],
             ubiquity['etf_count'][::-1], color='steelblue')
axes[0].set_xlabel('Number of ETFs')
axes[0].set_title('Top 20 Most Widely Held Stocks')

axes[1].barh(ubiquity['constituent_ticker'][::-1],
             ubiquity['avg_weight'][::-1], color='darkorange')
axes[1].set_xlabel('Average Weight % across ETFs')
axes[1].set_title('Avg Weight of Top 20 Stocks')

plt.suptitle('Most Widely Held Securities', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(ubiquity[['constituent_ticker', 'constituent_name', 'etf_count', 'avg_weight']].to_string(index=False))

In [ ]:
# Treemap: top 40 stocks by total market value across ETFs
top_mv = (
    con[con['market_value'].notna() & (con['market_value'] > 0)]
    .groupby(['constituent_ticker', 'constituent_name'])['market_value']
    .sum()
    .nlargest(40)
    .reset_index()
)
top_mv.columns = ['ticker', 'name', 'total_market_value']
top_mv['name_short'] = top_mv['name'].str[:25]

fig = px.treemap(
    top_mv, path=['name_short'], values='total_market_value',
    hover_data=['ticker'],
    title='Top 40 Securities by Aggregate Market Value Across All ETFs'
)
fig.show()

### 2.4 Country & Exchange Analysis

In [ ]:
country = con['country_of_exchange'].fillna('Unknown').value_counts().head(15).reset_index()
country.columns = ['country', 'count']

exchange = con['exchange'].fillna('Unknown').value_counts().head(15).reset_index()
exchange.columns = ['exchange', 'count']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(country['country'][::-1], country['count'][::-1],
             color=sns.color_palette('tab20', 15))
axes[0].set_xlabel('Number of Holdings')
axes[0].set_title('Top 15 Countries of Exchange')

axes[1].barh(exchange['exchange'][::-1], exchange['count'][::-1],
             color=sns.color_palette('tab20', 15))
axes[1].set_xlabel('Number of Holdings')
axes[1].set_title('Top 15 Exchanges')

plt.suptitle('Geographic & Exchange Breakdown (Holdings)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Currency distribution of holdings
curr = con['currency_traded'].fillna('Unknown').value_counts().head(10).reset_index()
curr.columns = ['currency', 'count']

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(curr['currency'], curr['count'], color=sns.color_palette('tab10', len(curr)))
ax.set_xlabel('Currency Traded')
ax.set_ylabel('Number of Holdings')
ax.set_title('Holdings by Trading Currency')
plt.tight_layout()
plt.show()

### 2.5 Time-Series Analysis (4 Snapshots)

In [ ]:
# Load all 4 snapshot dates from CONSTITUENTS
dates_df = query("""
    SELECT as_of_date,
           COUNT(DISTINCT composite_ticker) AS etf_count,
           COUNT(DISTINCT constituent_ticker) AS stock_count,
           COUNT(*) AS total_holdings,
           AVG(weight) AS avg_weight,
           MEDIAN(weight) AS median_weight
    FROM ETF_DB.LOCAL_COPY.CONSTITUENTS
    GROUP BY as_of_date
    ORDER BY as_of_date
""")
dates_df.columns = dates_df.columns.str.lower()
print(dates_df.to_string(index=False))

In [ ]:
# Holdings-count changes across snapshots for ETFs present in all 4 dates
etfs_all_dates = (
    con_ts.groupby('composite_ticker')['as_of_date'].nunique()
    .pipe(lambda s: s[s == 4].index)
)
stable = con_ts[con_ts['composite_ticker'].isin(etfs_all_dates)].copy()
stable['as_of_date'] = pd.to_datetime(stable['as_of_date'])

# Median num_holdings per snapshot
median_holdings = stable.groupby('as_of_date')['num_holdings'].median().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(median_holdings['as_of_date'], median_holdings['num_holdings'],
             marker='o', color='steelblue')
axes[0].set_xlabel('Snapshot Date')
axes[0].set_ylabel('Median Holdings per ETF')
axes[0].set_title('Median Holdings per ETF Across Snapshots')
axes[0].tick_params(axis='x', rotation=20)

# Median total weight
median_tw = stable.groupby('as_of_date')['total_weight'].median().reset_index()
axes[1].plot(median_tw['as_of_date'], median_tw['total_weight'],
             marker='o', color='darkorange')
axes[1].set_xlabel('Snapshot Date')
axes[1].set_ylabel('Median Total Weight (%)')
axes[1].set_title('Median Total Weight per ETF Across Snapshots')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle(f'Time-Series Trends (ETFs present in all 4 snapshots: {len(etfs_all_dates):,})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Change in num_holdings: first vs last snapshot
first_date = stable['as_of_date'].min()
last_date  = stable['as_of_date'].max()

pivot = stable.pivot_table(index='composite_ticker', columns='as_of_date',
                            values='num_holdings', aggfunc='first')
pivot['delta'] = pivot[last_date] - pivot[first_date]
pivot = pivot.dropna(subset=['delta'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(pivot['delta'].clip(-50, 50), bins=60, color='teal', edgecolor='white')
ax.axvline(0, color='red', linestyle='--')
ax.set_xlabel(f'Change in Holdings Count ({first_date.date()} → {last_date.date()})')
ax.set_ylabel('Number of ETFs')
ax.set_title('Holdings Count Change (clipped ±50)')
plt.tight_layout()
plt.show()

print(f"ETFs with more holdings  : {(pivot['delta'] > 0).sum():,}")
print(f"ETFs with same holdings  : {(pivot['delta'] == 0).sum():,}")
print(f"ETFs with fewer holdings : {(pivot['delta'] < 0).sum():,}")

## 3. Combined Analysis — INDUSTRY + CONSTITUENTS

In [ ]:
# Join INDUSTRY onto CONSTITUENTS summary
con_summary = (
    con.groupby('composite_ticker')
    .agg(
        actual_holdings=('constituent_ticker', 'count'),
        total_weight=('weight', 'sum'),
        top10_weight=('weight', lambda s: s.nlargest(10).sum()),
        total_market_value=('market_value', 'sum'),
    )
    .reset_index()
)
con_summary.columns.name = None

combined = ind.merge(con_summary, left_on='composite_ticker',
                     right_on='composite_ticker', how='inner')
combined['aum_bn'] = combined['aum'] / 1e9
combined['net_expenses_bps'] = combined['net_expenses'] * 100

print(f"Combined: {len(combined):,} ETFs matched")

In [ ]:
# Reported num_holdings vs actual holdings count
match = combined[combined['num_holdings'].notna() & (combined['num_holdings'] > 0)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(match['num_holdings'], match['actual_holdings'],
                alpha=0.2, s=10, color='steelblue')
lim = max(match['num_holdings'].max(), match['actual_holdings'].max())
axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1)
axes[0].set_xlabel('Reported Holdings (INDUSTRY.num_holdings)')
axes[0].set_ylabel('Actual Holdings (CONSTITUENTS count)')
axes[0].set_title('Reported vs Actual Holdings Count')

diff = match['actual_holdings'] - match['num_holdings']
axes[1].hist(diff.clip(-100, 100), bins=60, color='darkorange', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Actual − Reported (clipped ±100)')
axes[1].set_title('Holdings Count Discrepancy')

plt.suptitle('Data Quality: Reported vs Actual Holdings', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# AUM vs actual holdings count by asset class
plot_df = combined[combined['aum_bn'].notna() & (combined['aum_bn'] > 0)].copy()

fig = px.scatter(
    plot_df, x='actual_holdings', y='aum_bn',
    color='asset_class',
    log_x=True, log_y=True,
    hover_data=['composite_ticker', 'description', 'issuer'],
    opacity=0.5,
    labels={'actual_holdings': 'Actual Holdings (log)', 'aum_bn': 'AUM $B (log)'},
    title='AUM vs Holdings Count by Asset Class'
)
fig.update_traces(marker_size=5)
fig.show()

In [ ]:
# Concentration (top-10 weight) vs AUM
conc_df = combined[
    combined['aum_bn'].notna() & (combined['aum_bn'] > 0) &
    combined['top10_weight'].notna()
].copy()

fig = px.scatter(
    conc_df, x='aum_bn', y='top10_weight',
    color='asset_class', opacity=0.4,
    log_x=True,
    hover_data=['composite_ticker', 'description'],
    labels={'aum_bn': 'AUM $B (log)', 'top10_weight': 'Top-10 Holdings Weight (%)'},
    title='Portfolio Concentration vs AUM'
)
fig.update_traces(marker_size=5)
fig.show()

In [ ]:
# Expense ratio vs AUM by asset class (box plots)
box_df = combined[
    combined['net_expenses_bps'].notna() & (combined['net_expenses_bps'] > 0) &
    combined['asset_class'].notna()
].copy()

fig = px.box(
    box_df, x='asset_class', y='net_expenses_bps',
    color='asset_class',
    points=False,
    labels={'net_expenses_bps': 'Net Expense Ratio (bps)', 'asset_class': 'Asset Class'},
    title='Expense Ratio Distribution by Asset Class'
)
fig.update_yaxes(range=[0, 200])
fig.show()

In [ ]:
# Summary statistics table by asset class
summary = (
    combined[combined['aum_bn'].notna() & (combined['aum_bn'] > 0)]
    .groupby('asset_class')
    .agg(
        etf_count=('composite_ticker', 'count'),
        total_aum_bn=('aum_bn', 'sum'),
        median_aum_bn=('aum_bn', 'median'),
        median_holdings=('actual_holdings', 'median'),
        median_expense_bps=('net_expenses_bps', 'median'),
        median_top10_weight=('top10_weight', 'median'),
    )
    .round(2)
    .sort_values('total_aum_bn', ascending=False)
)

print("=== Summary by Asset Class ===")
print(summary.to_string())

## 4. Key Findings Summary

In [ ]:
total_aum = ind['aum'].sum() / 1e12
top3_issuers = issuer.head(3)
top3_aum_pct = top3_issuers['total_aum_bn'].sum() / (ind['aum'].sum() / 1e9) * 100

print("══════════════════════════════════════")
print("  ETF UNIVERSE — KEY FINDINGS")
print("══════════════════════════════════════")
print(f"  ETFs in universe       : {len(ind):,}")
print(f"  Total AUM              : ${total_aum:.1f}T")
print(f"  Top-3 issuers share AUM: {top3_aum_pct:.1f}% (iShares, Vanguard, SSgA)")
print(f"  Equity ETF share       : {len(ind[ind['asset_class']=='Equity'])/len(ind)*100:.1f}%")
print()
print("HOLDINGS (latest snapshot 2025-04-02)")
print(f"  Constituent rows       : {len(con):,}")
print(f"  Unique securities      : {con['constituent_ticker'].nunique():,}")
print(f"  Median holdings/ETF    : {holdings_per_etf.median():.0f}")
print(f"  Median top-10 weight   : {etf_concentration.median():.1f}%")
print()
print("DATA QUALITY")
print(f"  ETFs in both tables    : {len(combined):,} / {len(ind):,}")
print("══════════════════════════════════════")

In [ ]:
conn.close()
print("Connection closed.")